In [ ]:
### Time frames - Define the start and end dates of the data that you want to look at (e.g. all 29 years between 1996 and 2025) (ICMECAT only goes to mid-2025 for now)
dataTimeframe = ["1996-01-01 00:00", "2025-01-01 00:00"]
start = dataTimeframe[0] # start date of data
end = dataTimeframe[1] # end date of data
# dataSourcesToUse = ["Wind"]
fileName = "Wind_" + str(start)[0:10] + "-" + str(end)[0:10]+ "_InputOutput.csv"

slowCutOff = 400
fastCutOff = 500

cmeBuffer = 12

# Get imports and functions needed
# from sunpy.net import Fido
# from sunpy.net import attrs as a
# from sunpy.timeseries import TimeSeries 
from sunpy.timeseries import GenericTimeSeries
# from sunpy.time import parse_time
# from sunpy.data import cache
# from sunpy.coordinates import spice, frames
# from sunpy.coordinates import get_horizons_coord, get_body_heliographic_stonyhurst
# from astropy import units as u
# from astropy.time import Time
import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
import numpy as np 
# from datetime import datetime
from datetime import timedelta
import os
# import urllib.request
import pandas as pd
# import datetime as dt
# from scipy.stats import norm
# import seaborn as sns
# import polars as pl
# from scipy import stats
# import scipy.stats as stats
# from scipy.stats import anderson
import calendar
import warnings

import astropy.units as u
import astropy.constants as const
from astropy.visualization import quantity_support
warnings.filterwarnings("ignore")


# Wind

## Read in dataframe

In [ ]:
windData = pd.read_csv(fileName, index_col=["Unnamed: 0"])

In [ ]:
print(windData.head(10))

## Add parameters

In [ ]:
# B  = windData["ScalarB"].values * u.nT
Bx = windData["Bx_GSEGSM"].values* u.nT
By = windData["ByGSE"].values * u.nT
Bz = windData["BzGSE"].values* u.nT
# v  = windData["FlowSpeed"].values * u.km / u.s
Vx  = windData["Vx_GSE"].values * u.km / u.s
Vy  = windData["Vy_GSE"].values * u.km / u.s
Vz  = windData["Vz_GSE"].values * u.km / u.s
n  = windData["ProtonDensity"].values * u.cm**-3
T  = windData["Temp"].values * u.K

In [ ]:
# total solar wind speed
TotVsw = np.sqrt(Vx**2 + Vy**2 + Vz**2)

# magnitude of magnetic field
BMag = np.sqrt(Bx**2 + By**2 + Bz**2)

#Alfvén speed
vA = BMag / np.sqrt(const.mu0 * n * const.m_p)
vA = vA.to(u.km / u.s)

#Alfvén mach number
Ma = (TotVsw / vA).to(u.dimensionless_unscaled)

#Dynamic pressure
Pdyn = (n * const.m_p * TotVsw**2).to(u.nPa)
# Pdyn_by_hand = 1.6726e-6 * windData["ProtonDensity"] * windData["FlowSpeed"]**2 

#Beta
beta = (n * const.k_B * T / (BMag**2 / (2 * const.mu0))).to(u.dimensionless_unscaled)

#Cone and clock angles
cone = np.arccos(np.clip(Bx / BMag, -1, 1)).to(u.deg)    
clock = np.arctan2(By, Bz).to(u.deg)  



# spacecraft locations
R = 1 * u.au # for now set to 1 au, later use some conversion from -> windData["X_s_c_GSE"], windData["Y_s_c_GSE"], windData["Z_s_c_GSE"]
lat = 0 * u.deg # for now set to 0 degrees, later use some conversion from -> windData["X_s_c_GSE"], windData["Y_s_c_GSE"], windData["Z_s_c_GSE"]

In [ ]:
windData["TotVsw"]    = TotVsw.to_value(u.km / u.s)
windData["BMag"]      = BMag.to_value(u.nT)
windData["vA_kms"]    = vA.to_value(u.km / u.s)
windData["Ma"]        = Ma.value
windData["Pdyn_nPa"]  = Pdyn.to_value(u.nPa)
windData["beta"]      = beta.value
windData["cone_deg"]  = cone.to_value(u.deg)
windData["clock_deg"] = clock.to_value(u.deg)

windData["r"]  = R.to_value(u.au)
windData["lat"] = lat.to_value(u.deg)

windData.describe().round(2)

In [ ]:
# quantity_support()
# fig, axes = plt.subplots(5, 1, figsize=(9, 9), sharex=True)
# axes[0].plot(windData.index, BMag)
# axes[1].plot(windData.index, TotVsw)
# axes[2].plot(windData.index, Pdyn)
# axes[3].plot(windData.index, beta)
# axes[3].set_yscale("log")
# axes[4].plot(windData.index, Ma)

# for ax, name in zip(axes, ["B", "v", "Pdyn", "beta", "Ma"]):
#     ax.set_title(name, loc="left", fontsize=9)

# fig.tight_layout()

## Add CME column

In [ ]:
## reading the catalogue
icmes = pd.read_csv("HELIO4CAST_ICMECAT_v23.csv")
icmes_wind = icmes[icmes["sc_insitu"]=="Wind"]
print(icmes_wind)

## just making these times more consistent to be like the df ones
## the .tz_localise to to remove the fact that the catalogue has the datatimes Z at the end.
for column in ["icme_start_time", "mo_start_time", "mo_end_time"]:
    icmes_wind[column] = pd.to_datetime(icmes_wind[column]).dt.tz_localize(None)

## here what we can do is create an ICME column with the same index as the dataframe. 
## then set it all to 0, then look at the time ranges from the catalogue and set 
## this column equal to sheath as 1, cme (magnetic object as 2), and then the rest as 0. you can also just change the start times and end times here etc. 
icme_column = pd.Series(np.int8(0), index=pd.to_datetime(windData.index))

print(icme_column)
for ICMEstart, ICMEend in zip(icmes_wind["icme_start_time"]-timedelta(hours=cmeBuffer), icmes_wind["mo_start_time"]):
    icme_column.loc[ICMEstart:ICMEend] = 1
for ICMEstart, ICMEend in zip(icmes_wind["mo_start_time"], icmes_wind["mo_end_time"]+timedelta(hours=cmeBuffer)):
    icme_column.loc[ICMEstart:ICMEend] = 2

print(icme_column)

windData["cmes"] = icme_column
print(windData["cmes"])

#Laura's note
# you can also change the icme_start_time to be icme_start_time - 12 hours etc. this is just directly from the times from the catalogue.
# then you can do nice things like:
# no_cmes = df.where(df["cmes"] == 0)
# or no_cmes = df[df["cmes"]==0] etc.

In [ ]:
# icmes_wind["icme_start_time"].iloc[532:540], icmes_wind["mo_start_time"][532:540]
# windData[0:int(365.25*24*60 + 1080)]
# windData[0:int(365.25*24*60 + 1080)]["cmes"].unique()
# icme_column[0:int(365.25*24*60 + 1080)]
# icme_column[0:int(365.25*24*60 + 1080)].unique()

## Add sw type column

In [ ]:
def wind_type(speed):
    if np.isnan(speed):
        return "no data"
    elif speed > fastCutOff:
        return "fast"
    elif speed > slowCutOff:
        return "intermediate"
    else:
        return "slow"

windData["sw_type"] = windData["TotVsw"].apply(wind_type)

## Save to csv

In [ ]:
windData.to_csv("Wind_" + str(start)[0:10] + "-" + str(end)[0:10]+ "_Enriched.csv")